# Step 4 — support mass as a readout-validity measure (one combined session)

Runs **both** pre-registrations against the existing 8 adapters, forward passes only:

1. `valence_position_check.py` — the registered valence position-contamination check
   (`docs/valence-check-prereg-2026-07-29.md`, D1–D4 + P1), executed **unmodified**.
2. `step4_run.py` — `docs/step4-prereg-support-mass-2026-07-29.md` (R2, gated
   2026-07-29): the ΔSupportMass primary on three readouts, Q-A/Q-B/Q-C, B5, the
   prior-regression diagnostic, and the branch verdict, all evaluated mechanically
   from thresholds fixed before this session.

**STOP — one gate before running.** The prereg's **§9 Amendment log** records two
post-gate, pre-run amendments (R2.1 scale/convention pins; R2.2 the two-turn rating
position). If you have not read §9, read it before executing a single cell —
pressing Run is the gate on those amendments.

Prerequisites:
- Copy to `MyDrive/phi-map/step4-launch/`: `step4_run.py`,
  `valence_position_check.py`, `equanimity_factorial/train_eval.py`.
- An `HF_TOKEN` Colab secret with Llama-3.1-8B-Instruct access (never paste a token).
- **Runtime → Change runtime type → A100 GPU.**

Estimate (not yet spent): ~5–10 min for the check, plus generations for Q-C —
8×200 AdvBench, 9×24 probe responses, 8×24 rating turns — **order 1–1.5 A100-hours
total**. Everything lands on Drive under `equanimity-factorial-v1/step4/`.
Reads `evals/` and `adapters/` only; writes nothing outside `step4/`;
`quarantine/` is never touched.

In [ ]:
# Colab supplies torch/CUDA. Pins match the repo's other notebooks.
print("Protocol: STEP4_SUPPORT_MASS_V1 + VALENCE_POSITION_CHECK (registered)")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" "peft==0.18.0" "datasets" "scipy"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/step4-launch"
STAGE = {
    "step4_run.py": "/content/step4_run.py",
    "valence_position_check.py": "/content/valence_position_check.py",
    "train_eval.py": "/content/equanimity_factorial/train_eval.py",
}
os.makedirs("/content/equanimity_factorial", exist_ok=True)
for name, dest in STAGE.items():
    src = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(src), f"Missing {src} — copy it to step4-launch/ first"
    shutil.copy2(src, dest)
print("Step 4 launch files staged: OK")

In [ ]:
import glob, os

WORK = "/content/drive/MyDrive/phi-map/equanimity-factorial-v1"

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

from huggingface_hub import hf_hub_download
hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

adapters = sorted(p for p in glob.glob(f"{WORK}/adapters/*") if os.path.isdir(p))
assert len(adapters) == 8, f"expected 8 adapters, found {len(adapters)}"
assert os.path.exists(f"{WORK}/valence_direction.npz"), "frozen direction missing"
print(f"WORK OK: 8 adapters, direction present")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic estimator tests BEFORE any GPU spend: MDE arithmetic, the exact
# 8! permutation null (planted / nothing-works / independent worlds), the Δ rule,
# marker slicing, Q-C conventions, and the n=7 clustering of the descriptive Q-B.
import os, subprocess, sys
env = dict(os.environ, HF_TOKEN=HF_TOKEN)
subprocess.run([sys.executable, "/content/step4_run.py", "--self-test"],
               check=True, env=env)

In [ ]:
# 1/2 — the registered valence check, exactly as pre-registered. Its D1–D4 and
# P1 criteria were fixed in docs/valence-check-prereg-2026-07-29.md before any
# hidden state was seen; the script evaluates them mechanically.
import os, shutil, subprocess, sys
env = dict(os.environ, HF_TOKEN=HF_TOKEN)
subprocess.run([sys.executable, "/content/valence_position_check.py",
                "--work", WORK], check=True, env=env)
os.makedirs(f"{WORK}/step4", exist_ok=True)
shutil.copy2("/content/valence_position_check.json",
             f"{WORK}/step4/valence_position_check.json")
print("valence check artifact copied to Drive: OK")

In [ ]:
# 2/2 — the Step 4 measurement and pre-registered analysis. Prints per-readout
# Δ/CI/floors, Q-A, Q-B (descriptive + prior-regression with the exact 8! null),
# Q-C with coverage, B5, and the branch verdict. ~1–1.5 h, dominated by the
# 8×200 AdvBench generations.
import subprocess, sys
env = dict(os.environ, HF_TOKEN=HF_TOKEN)
subprocess.run([sys.executable, "/content/step4_run.py", "--work", WORK],
               check=True, env=env)

## After the run

Artifacts on Drive under `equanimity-factorial-v1/step4/`:
`step4_results.json` (all measurements + the mechanical analysis),
`step4_generations.json` (every generation, re-scorable offline),
`valence_position_check.json` (the registered check).

**The branch printed by `step4_run.py` is the verdict.** The write-up reports it —
it does not re-derive it, soften it, or swap an endpoint after seeing which way it
went. Everything differential-by-condition stays exploratory (prereg §4), the
estimand is the fixed mixture over these 8 adapters (§3), and a supported
prior-regression licenses the gate claim for renormalised-tail readouts only
(§2 Q-B scope). NOT SUPPORTED at 8 clusters is reported as undetected, not absence.